# Machine Learning - Previsão de Mortes Violentas

Este notebook treina modelos de regressão para prever a taxa de mortes violentas intencionais por 100 mil habitantes nas capitais brasileiras.

Fonte oficial:

`datamart.vw_base_modelagem_ml`

Estratégia metodológica:

- **Linear Regression**: baseline simples e interpretável.
- **Ridge Regression**: baseline linear regularizado.
- **Ridge com target logarítmico**: alternativa para reduzir previsões negativas.
- **Random Forest Regressor**: candidato principal por capturar relações não lineares.

A regressão linear será mantida como baseline comparativo, não como modelo final obrigatório. A comparação entre modelos ajuda a avaliar se indicadores educacionais e modelos não lineares melhoram a previsão.

## 1. Imports

In [1]:
import os

import numpy as np
import pandas as pd

from sqlalchemy import create_engine

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 2. Conexão com o PostgreSQL

Dentro do Docker Compose, o host do PostgreSQL é `postgres-service`.

Se o notebook for executado fora do container, altere `POSTGRES_HOST` para `localhost`.

In [2]:
DB_USER = os.getenv('POSTGRES_USER', 'postgres')
DB_PASSWORD = os.getenv('POSTGRES_PASSWORD', 'postgres')
DB_HOST = os.getenv('POSTGRES_HOST', 'postgres-service')
DB_PORT = os.getenv('POSTGRES_PORT', '5432')
DB_NAME = os.getenv('POSTGRES_DB', 'seguranca_publica')

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

## 3. Carregar base de modelagem

A view abaixo foi criada para Machine Learning. Ela já traz o target do ano atual e variáveis criminais defasadas (`lag1`) para evitar vazamento de informação.

In [3]:
query = """
SELECT *
FROM datamart.vw_base_modelagem_ml
ORDER BY codigo_municipio, ano;
"""

df_raw = pd.read_sql(query, engine)
df_raw.head()

,ano,codigo_municipio,nome_municipio,sigla_uf,nome_regiao,populacao_total,populacao_crescimento_pct,idhm,idhm_renda,idhm_educacao,...,ideb,fluxo,aprendizado,nota_mt,nota_lp,taxa_crimes_100k_lag1,taxa_mortes_violentas_100k_lag1,risco_indice_lag1,target_taxa_mortes_violentas_100k,target_taxa_crimes_100k
0,2016,1100205,Porto Velho,RO,Norte,499293.0,NaN,0.736,0.764,0.638,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.254091,NaN
1,2017,1100205,Porto Velho,RO,Norte,509323.0,2.0088,0.736,0.764,0.638,...,4.0,0.8735,4.5271,272.03,268.33,NaN,38.254091,0.622802,31.021572,NaN
2,2018,1100205,Porto Velho,RO,Norte,519531.0,2.0042,0.736,0.764,0.638,...,NaN,NaN,NaN,NaN,NaN,NaN,31.021572,0.544321,30.797007,772.427439
3,2019,1100205,Porto Velho,RO,Norte,529544.0,1.9273,0.736,0.764,0.638,...,4.3,0.8947,4.7708,278.81,278.25,772.427439,30.797007,0.487566,22.472165,851.865001
4,2020,1100205,Porto Velho,RO,Norte,539354.0,1.8525,0.736,0.764,0.638,...,NaN,NaN,NaN,NaN,NaN,851.865001,22.472165,0.356798,29.850525,NaN


## 4. Diagnóstico inicial

In [4]:
display(df_raw.shape)
display(df_raw.dtypes)
display(df_raw.isna().sum().sort_values(ascending=False))
display(df_raw.groupby('ano').size().rename('qtd_linhas'))

(243, 21)

ano                                    int64
codigo_municipio                       int64
nome_municipio                        object
sigla_uf                              object
nome_regiao                           object
populacao_total                      float64
populacao_crescimento_pct            float64
idhm                                 float64
idhm_renda                           float64
idhm_educacao                        float64
idhm_longevidade                     float64
ideb                                 float64
fluxo                                float64
aprendizado                          float64
nota_mt                              float64
nota_lp                              float64
taxa_crimes_100k_lag1                float64
taxa_mortes_violentas_100k_lag1      float64
risco_indice_lag1                    float64
target_taxa_mortes_violentas_100k    float64
target_taxa_crimes_100k              float64
dtype: object

target_taxa_crimes_100k              141
taxa_crimes_100k_lag1                141
ideb                                 136
nota_lp                              136
nota_mt                              136
aprendizado                          136
fluxo                                136
risco_indice_lag1                     27
taxa_mortes_violentas_100k_lag1       27
populacao_crescimento_pct             27
target_taxa_mortes_violentas_100k      0
ano                                    0
codigo_municipio                       0
idhm_educacao                          0
idhm_renda                             0
idhm                                   0
populacao_total                        0
nome_regiao                            0
sigla_uf                               0
nome_municipio                         0
idhm_longevidade                       0
dtype: int64

ano
2016    27
2017    27
2018    27
2019    27
2020    27
2021    27
2022    27
2023    27
2024    27
Name: qtd_linhas, dtype: int64

## 5. Preparação da base

Como o IDEB não é divulgado todos os anos, usamos `forward fill` por UF para propagar o último valor educacional conhecido.

Isso não usa informação futura: cada ano recebe apenas o último valor disponível até aquele momento.

In [5]:
df = df_raw.copy()
df = df.sort_values(['sigla_uf', 'ano']).reset_index(drop=True)

colunas_numericas = [
    'ano',
    'populacao_total',
    'populacao_crescimento_pct',
    'idhm',
    'idhm_renda',
    'idhm_educacao',
    'idhm_longevidade',
    'ideb',
    'fluxo',
    'aprendizado',
    'nota_mt',
    'nota_lp',
    'taxa_crimes_100k_lag1',
    'taxa_mortes_violentas_100k_lag1',
    'risco_indice_lag1',
    'target_taxa_mortes_violentas_100k',
    'target_taxa_crimes_100k',
]

for coluna in colunas_numericas:
    df[coluna] = pd.to_numeric(df[coluna], errors='coerce')

colunas_educacao = ['ideb', 'fluxo', 'aprendizado', 'nota_mt', 'nota_lp']

for coluna in colunas_educacao:
    df[f'{coluna}_preenchido'] = df.groupby('sigla_uf')[coluna].ffill()

df[['ano', 'sigla_uf', 'ideb', 'ideb_preenchido']].head(12)

,ano,sigla_uf,ideb,ideb_preenchido
0,2016,AC,NaN,NaN
1,2017,AC,3.8,3.8
2,2018,AC,NaN,3.8
3,2019,AC,3.9,3.9
4,2020,AC,NaN,3.9
5,2021,AC,4.0,4.0
6,2022,AC,NaN,4.0
7,2023,AC,4.0,4.0
8,2024,AC,NaN,4.0
9,2016,AL,NaN,NaN


## 6. Definição de features e target

In [6]:
# Target principal: taxa de mortes violentas intencionais por 100 mil habitantes.
# Esse indicador tem cobertura completa para 2016-2024 nas capitais.
target = 'target_taxa_mortes_violentas_100k'

features_baseline = [
    'ano',
    'populacao_total',
    'populacao_crescimento_pct',
    'idhm',
    'idhm_renda',
    'idhm_educacao',
    'idhm_longevidade',
    'taxa_mortes_violentas_100k_lag1',
    'risco_indice_lag1',
]

features_educacao = features_baseline + [
    'ideb_preenchido',
    'fluxo_preenchido',
    'aprendizado_preenchido',
    'nota_mt_preenchido',
    'nota_lp_preenchido',
]

colunas_identificacao = ['ano', 'codigo_municipio', 'nome_municipio', 'sigla_uf', 'nome_regiao']

features_baseline, features_educacao

(['ano',
  'populacao_total',
  'populacao_crescimento_pct',
  'idhm',
  'idhm_renda',
  'idhm_educacao',
  'idhm_longevidade',
  'taxa_mortes_violentas_100k_lag1',
  'risco_indice_lag1'],
 ['ano',
  'populacao_total',
  'populacao_crescimento_pct',
  'idhm',
  'idhm_renda',
  'idhm_educacao',
  'idhm_longevidade',
  'taxa_mortes_violentas_100k_lag1',
  'risco_indice_lag1',
  'ideb_preenchido',
  'fluxo_preenchido',
  'aprendizado_preenchido',
  'nota_mt_preenchido',
  'nota_lp_preenchido'])

## 7. Split temporal

Plano ideal com a base atual carregada até 2024:

- Treino: 2017, 2018 e 2019
- Teste: 2023 e 2024
- Excluir: 2020, 2021 e 2022

Se esses anos ainda não existirem na base, o notebook usa uma divisão temporal alternativa para permitir validação durante o desenvolvimento.

In [7]:
if 'df' not in globals():
    raise RuntimeError('Execute primeiro as células de carga e preparação dos dados antes do split temporal.')

if 'target' not in globals() or 'features_baseline' not in globals() or 'features_educacao' not in globals():
    target = 'target_taxa_mortes_violentas_100k'
    features_baseline = [
        'ano',
        'populacao_total',
        'populacao_crescimento_pct',
        'idhm',
        'idhm_renda',
        'idhm_educacao',
        'idhm_longevidade',
        'taxa_mortes_violentas_100k_lag1',
        'risco_indice_lag1',
    ]
    features_educacao = features_baseline + [
        'ideb_preenchido',
        'fluxo_preenchido',
        'aprendizado_preenchido',
        'nota_mt_preenchido',
        'nota_lp_preenchido',
    ]
    colunas_identificacao = ['ano', 'codigo_municipio', 'nome_municipio', 'sigla_uf', 'nome_regiao']

anos_treino_planejado = [2017, 2018, 2019]
anos_teste_planejado = [2023, 2024]
anos_excluir = [2020, 2021, 2022]

def escolher_split_temporal(df_modelo, features, target):
    colunas_modelo = list(dict.fromkeys(colunas_identificacao + features + [target]))
    dados = df_modelo[colunas_modelo].dropna().copy()
    anos_disponiveis = set(dados['ano'].astype(int).unique())

    tem_split_planejado = (
        set(anos_treino_planejado).issubset(anos_disponiveis)
        and len(set(anos_teste_planejado).intersection(anos_disponiveis)) > 0
    )

    if tem_split_planejado:
        treino = dados[dados['ano'].isin(anos_treino_planejado)].copy()
        teste = dados[dados['ano'].isin(anos_teste_planejado)].copy()
        estrategia = 'split_planejado_pos_pandemia'
    else:
        dados_sem_anos_excluidos = dados[~dados['ano'].isin(anos_excluir)].copy()
        anos_validos = sorted(dados_sem_anos_excluidos['ano'].astype(int).unique())

        if len(anos_validos) >= 2:
            ano_teste = anos_validos[-1]
            treino = dados_sem_anos_excluidos[dados_sem_anos_excluidos['ano'] < ano_teste].copy()
            teste = dados_sem_anos_excluidos[dados_sem_anos_excluidos['ano'] == ano_teste].copy()
            estrategia = f'fallback_temporal_teste_{ano_teste}'
        else:
            anos_validos = sorted(dados['ano'].astype(int).unique())
            ano_teste = anos_validos[-1]
            treino = dados[dados['ano'] < ano_teste].copy()
            teste = dados[dados['ano'] == ano_teste].copy()
            estrategia = f'fallback_com_pandemia_teste_{ano_teste}'

    return treino, teste, estrategia

treino_a, teste_a, estrategia_a = escolher_split_temporal(df, features_baseline, target)
treino_b, teste_b, estrategia_b = escolher_split_temporal(df, features_educacao, target)

resumo_split = pd.DataFrame([
    {
        'modelo': 'Modelo A - Baseline sem educacao',
        'estrategia': estrategia_a,
        'anos_treino': sorted(treino_a['ano'].unique()),
        'anos_teste': sorted(teste_a['ano'].unique()),
        'linhas_treino': len(treino_a),
        'linhas_teste': len(teste_a),
    },
    {
        'modelo': 'Modelo B - Linear com educacao',
        'estrategia': estrategia_b,
        'anos_treino': sorted(treino_b['ano'].unique()),
        'anos_teste': sorted(teste_b['ano'].unique()),
        'linhas_treino': len(treino_b),
        'linhas_teste': len(teste_b),
    },
])

resumo_split

,modelo,estrategia,anos_treino,anos_teste,linhas_treino,linhas_teste
0,Modelo A - Baseline sem educacao,split_planejado_pos_pandemia,"[2017, 2018, 2019]","[2023, 2024]",81,54
1,Modelo B - Linear com educacao,split_planejado_pos_pandemia,"[2017, 2018, 2019]","[2023, 2024]",81,54


## 8. Funções de treino e avaliação

In [8]:
def treinar_e_avaliar(
    nome_modelo,
    estimador,
    treino,
    teste,
    features,
    target,
    usar_scaler=True,
    transformacao_target=None,
    limitar_previsao_zero=False,
):
    X_train = treino[features]
    y_train = treino[target]
    X_test = teste[features]
    y_test = teste[target]

    etapas = []
    if usar_scaler:
        etapas.append(('scaler', StandardScaler()))
    etapas.append(('model', estimador))

    pipeline = Pipeline(etapas)

    if transformacao_target == 'log1p':
        y_train_modelo = np.log1p(y_train)
    else:
        y_train_modelo = y_train

    pipeline.fit(X_train, y_train_modelo)
    y_pred = pipeline.predict(X_test)

    if transformacao_target == 'log1p':
        y_pred = np.expm1(y_pred)

    previsoes_negativas = int((y_pred < 0).sum())

    if limitar_previsao_zero:
        y_pred = np.maximum(y_pred, 0)

    metricas = {
        'modelo': nome_modelo,
        'mae': mean_absolute_error(y_test, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
        'r2': r2_score(y_test, y_pred) if len(y_test) > 1 else np.nan,
        'previsoes_negativas': previsoes_negativas,
        'linhas_treino': len(treino),
        'linhas_teste': len(teste),
    }

    previsoes = teste[colunas_identificacao].copy()
    previsoes['modelo'] = nome_modelo
    previsoes['valor_real'] = y_test.values
    previsoes['valor_previsto'] = y_pred
    previsoes['erro'] = previsoes['valor_real'] - previsoes['valor_previsto']
    previsoes['erro_absoluto'] = previsoes['erro'].abs()

    modelo = pipeline.named_steps['model']
    if hasattr(modelo, 'coef_'):
        importancias = pd.DataFrame({
            'modelo': nome_modelo,
            'feature': features,
            'tipo_importancia': 'coeficiente',
            'valor': modelo.coef_,
        }).sort_values('valor', ascending=False)
    elif hasattr(modelo, 'feature_importances_'):
        importancias = pd.DataFrame({
            'modelo': nome_modelo,
            'feature': features,
            'tipo_importancia': 'feature_importance',
            'valor': modelo.feature_importances_,
        }).sort_values('valor', ascending=False)
    else:
        importancias = pd.DataFrame(columns=['modelo', 'feature', 'tipo_importancia', 'valor'])

    return pipeline, metricas, previsoes, importancias

## 9. Modelo A - Baseline sem educação

In [9]:
modelo_a, metricas_a, previsoes_a, coeficientes_a = treinar_e_avaliar(
    nome_modelo='Modelo A - Linear sem educacao',
    estimador=LinearRegression(),
    treino=treino_a,
    teste=teste_a,
    features=features_baseline,
    target=target,
)

pd.DataFrame([metricas_a])

,modelo,mae,rmse,r2,linhas_treino,linhas_teste
0,Modelo A - Linear sem educacao,35.213269,35.777103,-6.997654,81,54


## 10. Modelo B - Linear com educação

In [10]:
modelo_b, metricas_b, previsoes_b, coeficientes_b = treinar_e_avaliar(
    nome_modelo='Modelo B - Linear com educacao',
    estimador=LinearRegression(),
    treino=treino_b,
    teste=teste_b,
    features=features_educacao,
    target=target,
)

pd.DataFrame([metricas_b])

,modelo,mae,rmse,r2,linhas_treino,linhas_teste
0,Modelo B - Linear com educacao,144.237276,172.006939,-183.860531,81,54


## 11. Modelo B Ridge - Versão regularizada

A Ridge Regression ajuda a estabilizar o modelo quando existem variáveis correlacionadas, como IDHM, IDEB, aprendizado e notas.

In [11]:
modelo_b_ridge, metricas_b_ridge, previsoes_b_ridge, coeficientes_b_ridge = treinar_e_avaliar(
    nome_modelo='Modelo B - Ridge com educacao',
    estimador=Ridge(alpha=1.0),
    treino=treino_b,
    teste=teste_b,
    features=features_educacao,
    target=target,
)

pd.DataFrame([metricas_b_ridge])

,modelo,mae,rmse,r2,linhas_treino,linhas_teste
0,Modelo B - Ridge com educacao,31.861106,32.443866,-5.576843,81,54


## 12. Comparação dos modelos

In [12]:
metricas_modelos = pd.DataFrame([metricas_a, metricas_b, metricas_b_ridge])
metricas_modelos.sort_values('rmse')

,modelo,mae,rmse,r2,linhas_treino,linhas_teste
2,Modelo B - Ridge com educacao,31.861106,32.443866,-5.576843,81,54
0,Modelo A - Linear sem educacao,35.213269,35.777103,-6.997654,81,54
1,Modelo B - Linear com educacao,144.237276,172.006939,-183.860531,81,54


## 13. Previsões e maiores erros

In [13]:
previsoes_modelos = pd.concat(
    [previsoes_a, previsoes_b, previsoes_b_ridge],
    ignore_index=True,
)

previsoes_modelos.sort_values(['modelo', 'erro_absoluto'], ascending=[True, False]).head(20)

,ano,codigo_municipio,nome_municipio,sigla_uf,nome_regiao,modelo,valor_real,valor_previsto,erro,erro_absoluto
14,2023,3205309,Vitoria,ES,Sudeste,Modelo A - Linear sem educacao,32.071186,-15.802301,47.873487,47.873487
9,2024,2927408,Salvador,BA,Nordeste,Modelo A - Linear sem educacao,51.967202,5.539107,46.428095,46.428095
15,2024,3205309,Vitoria,ES,Sudeste,Modelo A - Linear sem educacao,27.712952,-18.101677,45.814629,45.814629
31,2024,2611606,Recife,PE,Nordeste,Modelo A - Linear sem educacao,39.050026,-6.020282,45.070308,45.070308
11,2024,2304400,Fortaleza,CE,Nordeste,Modelo A - Linear sem educacao,33.677593,-10.111852,43.789445,43.789445
41,2024,1100205,Porto Velho,RO,Norte,Modelo A - Linear sem educacao,36.513859,-7.071450,43.585309,43.585309
35,2024,4106902,Curitiba,PR,Sul,Modelo A - Linear sem educacao,16.181716,-26.844904,43.026620,43.026620
52,2023,1721000,Palmas,TO,Norte,Modelo A - Linear sem educacao,41.619988,-0.911818,42.531806,42.531806
6,2023,1600303,Macapa,AP,Norte,Modelo A - Linear sem educacao,65.204517,22.938248,42.266269,42.266269
27,2024,1501402,Belem,PA,Norte,Modelo A - Linear sem educacao,26.384828,-15.664840,42.049668,42.049668


## 14. Interpretação dos coeficientes

In [14]:
coeficientes_modelos = pd.concat(
    [coeficientes_a, coeficientes_b, coeficientes_b_ridge],
    ignore_index=True,
)

coeficientes_modelos

,modelo,feature,coeficiente
0,Modelo A - Linear sem educacao,idhm,100.136203
1,Modelo A - Linear sem educacao,taxa_mortes_violentas_100k_lag1,11.216821
2,Modelo A - Linear sem educacao,populacao_crescimento_pct,1.533025
3,Modelo A - Linear sem educacao,populacao_total,0.048228
4,Modelo A - Linear sem educacao,risco_indice_lag1,-0.216649
5,Modelo A - Linear sem educacao,ano,-5.388247
6,Modelo A - Linear sem educacao,idhm_longevidade,-17.232193
7,Modelo A - Linear sem educacao,idhm_educacao,-45.967909
8,Modelo A - Linear sem educacao,idhm_renda,-48.951557
9,Modelo B - Linear com educacao,nota_mt_preenchido,11153.736302


## 15. Conclusões iniciais

Pontos para interpretar após executar o notebook:

- O Modelo B linear reduziu MAE/RMSE em relação ao Modelo A?
- O Ridge melhorou a estabilidade em relação à regressão linear simples?
- Quais capitais tiveram maior erro absoluto?
- Os coeficientes fazem sentido com a hipótese do projeto?
- A inclusão de educação agregou poder preditivo ou apenas complexidade?
- Um modelo não linear, como Random Forest, melhora os resultados sem gerar previsões negativas?

Observação: com a base atual, o teste pós-pandemia usa 2023 e 2024. Quando dados de 2025 forem carregados no DW/Data Mart, este notebook pode ser reexecutado incluindo esse ano no conjunto de teste.